In [ ]:
import json
import os
import math
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import time
import pyarrow.dataset as ds
import random
import seaborn as sns
import stylo_metrix as sm

from collections import deque, Counter
from datasets import load_dataset
from dotenv import load_dotenv
from google import genai
from google.genai.types import EmbedContentConfig, HttpOptions, UploadFileConfig
from google.oauth2 import service_account
from scipy.spatial.distance import cdist
from tqdm.notebook import tqdm
load_dotenv()

In [ ]:
dataset = ds.dataset(
    "/datasets/ai/allenai/hub/datasets--allenai--WildChat-4.8M/snapshots/c827c6df8fcf008219ffaffa4d1dd77491099367/data",
    format="parquet",
)
df = pd.read_csv("wildchat_filtered_4o20240806_41mini20250414_device_deduped.csv")

In [ ]:
print(f"Number of conversations: {len(df)}")

# Count the number of unique hashed IPs
unique_ips = df["hashed_ip"].nunique()
print(f"Number of unique hashed IPs: {unique_ips}")

# Count the number of unique headers
unique_headers = df["header"].astype(str).nunique()
print(f"Number of unique headers: {unique_headers}")

# Count how many IPs exist for each row count
ip_counts = df["hashed_ip"].value_counts()
row_count_distribution = ip_counts.value_counts().sort_index()

# Print line by line: "Row count X → Y IPs"
print("\nNumber of IPs per num conversations:")
for rows, num_ips in row_count_distribution.items():
    print(f"{rows}: {num_ips} IPs")

# Count the frequency of each hashed IP
max_per_ip = 100
ip_counts_filtered = ip_counts[(ip_counts >= 1) & (ip_counts <= max_per_ip)]

# Plot histogram of IP frequencies
plt.figure(figsize=(18, 6))
bins = range(1, max_per_ip + 2)
plt.hist(ip_counts_filtered, bins=bins, edgecolor="black", alpha=0.7, log=True)
bin_centers = [b + 0.5 for b in bins[:-1]]
plt.xlim(1, max_per_ip + 2)
plt.xticks(bin_centers, bins[:-1], fontsize=10, rotation=-45)
plt.title("Histogram of IPs by Number of Conversations")
plt.xlabel("Number of conversations")
plt.ylabel("Number of IPs")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Count the number of records per unique combination
counts = df.groupby(['hashed_ip', 'header', 'language']).size().reset_index(name='count')

# Step 2: Inspect the counts (optional)
print(counts.sort_values("count", ascending=False).head())

# Step 3: Plot the distribution of the counts
plt.figure(figsize=(18,6))
max_bin = 100
bins = range(1, max_bin + 2)
plt.hist(counts['count'], bins=bins, color='skyblue', edgecolor='black', log=True)
bin_centers = [b + 0.5 for b in bins[:-1]]
plt.xlim(1, max_bin + 2)
plt.xticks(bin_centers, bins[:-1], fontsize=10, rotation=-45)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(counts.sort_values("count", ascending=False)["count"].to_list(), log=True, bins=20)
plt.xscale("log", base=10)
plt.show()

In [ ]:
# Model usage over time

model_ts = dataset.to_table(columns=["timestamp", "model"]).to_pandas()

# Ensure timestamp is datetime
model_ts['timestamp'] = pd.to_datetime(model_ts['timestamp'])

# Set timestamp as index
model_ts.set_index('timestamp', inplace=True)

# Count occurrences of each model per month
monthly_counts = model_ts.groupby('model').resample('ME').size().unstack(0).fillna(0)

# Plot as stacked area chart
num_models = monthly_counts.shape[1]
colors = cm.get_cmap('tab20', num_models)  # 'tab20' has 20 distinct colors

# Plot with custom colors
monthly_counts.plot(figsize=(14,7), color=[colors(i) for i in range(num_models)])
plt.title('Monthly Usage of Models Over Time')
plt.xlabel('Month')
plt.ylabel('Count')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1))  # put legend outside
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
model_ts.groupby("model").value_counts()

In [ ]:
# Build prefixes and suffix for filtered and deduped dataset
min_prefixes = {}
min_suffixes = {}

df = df.drop_duplicates("conversation")
for min_len in tqdm(range(10, 201, 10)):
    prefixes = {}
    suffixes = {}

    for idx, row in df.iterrows():
        conv = row["conversation"]
        identity = row["hashed_ip"] + "|" + row["accept_language"] + "|" + row["device_info"]
        model = row["model"]
        ts = row["timestamp"]
        text = conv.split("\n===\n", 1)[0]
        if len(text) < min_len:
            continue

        prefix = text[:min_len]
        if prefix in prefixes:
            prefixes[prefix]["indices"].append(idx)
            prefixes[prefix]["ids"].add(identity)
        else:
            prefixes[prefix] = {"indices": [idx], "ids": set([identity])}

        suffix = text[-min_len:]
        if suffix in suffixes:
            suffixes[suffix]["indices"].append(idx)
            suffixes[suffix]["ids"].add(identity)
        else:
            suffixes[suffix] = {"indices": [idx], "ids": set([identity])}
    
    min_prefixes[min_len] = prefixes
    min_suffixes[min_len] = suffixes

In [ ]:
plt.figure(figsize=(12, 6))
for min_num_indices in range(2, 15):
    y = [sum(l for v in prefixes.values() if (l := len(v["indices"])) >= min_num_indices) / len(df) for prefixes in min_prefixes.values()]
    plt.plot(list(min_prefixes.keys()), y, marker="o", label=f"{min_num_indices}")

plt.xticks(list(min_prefixes.keys()), rotation=45)
plt.grid(linestyle="--", color="lightgray", alpha=0.5)
plt.legend(title="Min # of conv.")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
for min_num_indices in range(2, 15):
    y = [sum(l for v in suffixes.values() if (l := len(v["indices"])) >= min_num_indices) / len(df) for suffixes in min_suffixes.values()]
    plt.plot(list(min_suffixes.keys()), y, marker="o", label=f"{min_num_indices}")

plt.xticks(list(min_suffixes.keys()), rotation=45)
plt.grid(linestyle="--", color="lightgray", alpha=0.5)
plt.legend(title="Min # of conv.")
plt.show()

In [ ]:
same_prefix_texts = [(k, l) for k, v in min_prefixes[50].items() if (l := len(v["indices"])) >= 5]
print(sum(t[1] for t in same_prefix_texts))
for k, l in sorted(same_prefix_texts, key=lambda x: x[1], reverse=True):
    print(f"{l}:{k}")

In [ ]:
same_suffix_texts = [(k, l) for k, v in min_suffixes[50].items() if (l := len(v["indices"])) < 5]
print(sum(t[1] for t in same_suffix_texts))
for k, l in sorted(same_suffix_texts, key=lambda x: x[1], reverse=True):
    print(f"{l}:{k}")

In [ ]:
count = {}
prefixes = min_prefixes[50]
for v in prefixes.values():
    l = len(v["indices"])
    count[l] = count.get(l, 0) + l
x = sorted(count.keys())[1:]
y = [(count[k]) for k in x]

plt.figure(figsize=(18, 6))
pos = range(len(x))
plt.bar(pos, y, width=0.5, log=True)
plt.xticks(pos, x, rotation=90)
plt.grid(linestyle="--", color="lightgray", alpha=0.5, zorder=999)
plt.show()

In [ ]:
count = {}
prefixes = min_prefixes[50]
for v in prefixes.values():
    l = len(v["ids"])
    count[l] = count.get(l, 0) + l
x = sorted(count.keys())
y = [(count[k]) for k in x]

plt.figure(figsize=(18, 6))
pos = range(len(x))
plt.bar(pos, y, width=0.5, log=True)
plt.xticks(pos, x, rotation=90)
plt.grid(linestyle="--", color="lightgray", alpha=0.5, zorder=999)
plt.show()

In [ ]:
x = sorted(count.keys())[1:]
# len_df = 3199860
len_df = len(df)
y = [count[k] / len_df for k in x]
plt.figure(figsize=(18, 6))
plt.plot(x, np.cumsum(y), marker=".")
plt.ylim(0.0, 0.6)
plt.xscale('log', base=10)
plt.grid(linestyle="--", color="lightgray", zorder=999)
plt.show()

In [ ]:
avg_ids_per_prefix = [len(v["ids"]) / len(v["indices"]) for v in prefixes.values() if len(v["indices"]) > 1]
plt.hist(avg_ids_per_prefix, bins=10)
plt.show()

In [ ]:
sum(len(v["indices"]) for v in prefixes.values() if len(v["indices"]) > 1 and len(v["ids"]) / len(v["indices"]) < 0.05)

In [ ]:
avg_ids_per_prefix_boxplot = {}
for v in min_prefixes[50].values():
    num_idx = len(v["indices"])
    if num_idx == 1:
        continue
    if num_idx not in avg_ids_per_prefix_boxplot:
        avg_ids_per_prefix_boxplot[num_idx] = []
    avg_ids_per_prefix_boxplot[num_idx].append(len(v["ids"]) / num_idx)

sorted_items = sorted(avg_ids_per_prefix_boxplot.items(), key=lambda x: (np.median(x[1]), np.quantile(x[1], 0.25), x[0]))
keys = [k for k, _ in sorted_items]
values = [v for _, v in sorted_items]

# --- Create plot
fig, ax = plt.subplots(figsize=(12, 12))  # tall figure

bp = ax.boxplot(
    values,
    vert=False,
    showfliers=False,
    patch_artist=True,
    # widths=0.2,
    # boxprops=dict(linewidth=0.8),
    # whiskerprops=dict(linewidth=0.6),
    # capprops=dict(linewidth=0.6),
    # medianprops=dict(linewidth=1.0),
)

# --- Styling (clean + readable)
for box in bp['boxes']:
    box.set_alpha(0.6)

ax.set_yticks(range(1, len(keys) + 1))
ax.set_yticklabels(keys, fontsize=6)

ax.set_xlabel("Value")
ax.set_title("Horizontal Boxplot (Sorted by Median)")

ax.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
print(df.groupby("hashed_ip")["header"].value_counts().sort_values(ascending=False).to_string())

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
for period in ["Y", "M", "D", "12h", "h"]:
    if "h" in period:
        df[f"timestamp_{period}"] = df["timestamp"].dt.floor(period)
    else:
        df[f"timestamp_{period}"] = df["timestamp"].dt.to_period(period)
    counts = (
        df.groupby(["hashed_ip", f"timestamp_{period}"])
            # ["turn"].sum()    # Turns
            .size()             # Conversations
            .groupby(["hashed_ip"])
            .mean()
            .sort_values(ascending=False)
    )
    counter = Counter([round(c) for c in counts.to_list()])
    total = sum(k * counter[k] for k in counter.keys())
    counter_keys = sorted(counter.keys())
    plt.plot(counter_keys, np.cumsum([k * counter[k]/total for k in counter_keys]), marker=".")
    plt.xscale("log")
    plt.grid(linestyle="--", color="lightgray")
    plt.title(period)
    plt.show()

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["month"] = df["timestamp"].dt.floor("M")

monthly_avg = (
    df.groupby(["hashed_ip", "month"])
      .size()
      .groupby(["hashed_ip",])
      .mean()
)

filtered = df.merge(
    monthly_avg[(monthly_avg < 100)],
    on=["hashed_ip",],
    how="inner"
)

In [ ]:
ts = pd.to_datetime(df["timestamp"])
monthly_avg = (
    df.assign(month=ts.dt.to_period("M"))
      .groupby(["hashed_ip", "month"])
      .size()
      .groupby("hashed_ip")
      .mean()
)

filtered = df[df["hashed_ip"].isin(monthly_avg[monthly_avg > 100].index)]
print(filtered.sort_values(["hashed_ip", "timestamp"]).head(100)[["timestamp", "hashed_ip", "model", "header", "conversation"]].to_string())

In [ ]:
# Prepare data for linkage analysis

use_stylo = True
df = pd.read_csv("wildchat_filtered_4o20240806_41mini20250414_device_deduped.csv")

# Optional filtering
lang = ("English", "en")
# lang = ("Russian", "ru")

df = df[df["language"] == lang[0]]
df = df[df.groupby(["hashed_ip", "accept_language", "device_info"])["model"]
    .transform("nunique")
    .ge(2)
]
df = df.sort_values(["hashed_ip", "accept_language", "device_info", "timestamp"])

# Split
df["identity"] = df["hashed_ip"] + "|" + df["accept_language"] + "|" + df["device_info"]
known_filter = df["model"] == "gpt-4o-2024-08-06"
unknown_filter = df["model"] == "gpt-4.1-mini-2025-04-14"
known = df[known_filter].reset_index(drop=True)
unknown = df[unknown_filter].reset_index(drop=True)
known_ids = known["identity"]
unknown_ids = unknown["identity"]
num_unique_ids = len(set(known_ids))

known_ids_to_conv_idx = {}
for known_id in known_ids:
    known_ids_to_conv_idx[known_id] = known[known["identity"] == known_id].index.tolist()

unknown_ids_to_conv_idx = {}
for unknown_id in unknown_ids:
    unknown_ids_to_conv_idx[unknown_id] = unknown[unknown["identity"] == unknown_id].index.tolist()

unknown_ids_conv_count = unknown.groupby(["identity"]).size().to_dict()

if use_stylo:
    embeddings = pd.read_csv(f"wildchat_embeddings/wildchat_filtered_{lang[1]}_2048_stylometrix.csv").drop(columns="text")
    known_emb = embeddings[known_filter.reset_index(drop=True)].to_numpy()
    unknown_emb = embeddings[unknown_filter.reset_index(drop=True)].to_numpy()
else:
    embeddings = np.load("wildchat_embeddings/wildchat_gemini-embedding-001.npy")
    known_emb = embeddings[known["idx"].to_list()]
    unknown_emb = embeddings[unknown["idx"].to_list()]
    known_emb = known_emb / np.linalg.norm(known_emb, axis=1, keepdims=True)
    unknown_emb = unknown_emb / np.linalg.norm(unknown_emb, axis=1, keepdims=True)

del embeddings

In [ ]:
# Embedding only

if use_stylo:
    similarity = -cdist(unknown_emb, known_emb, metric='euclidean')    
else:
    similarity = unknown_emb @ known_emb.T

sim_rank = np.argsort(-similarity, axis=1)
sim_rank_correct = []
sim_rank_ids = []

for i, rank in enumerate(sim_rank):
    target_id = unknown_ids.iloc[i]
    rank_ids = known_ids.iloc[rank]
    sim_rank_correct.append((rank_ids == target_id).to_list())
    sim_rank_ids.append(list({id: True for id in rank_ids}.keys())) # Dict is ordered by insertion

In [ ]:
def analyze_wildchat(
    sim_rank_correct,
    sim_rank_ids,
    unknown,
    unknown_ids_conv_count,
    sample=None,
):
    if sample == None:        
        valid_ids = set(unknown_ids)
        sample = len(valid_ids)
    else:
        assert isinstance(sample, int) and sample > 0
        valid_ids = set(random.sample(list(set(unknown_ids.to_list())), sample))
    valid_idx = [idx for valid_id in valid_ids for idx in unknown_ids_to_conv_idx[valid_id]]
    sim_rank_correct = [sim_rank_correct[idx] for idx in valid_idx]
    sim_rank_ids = [[iden for iden in sim_rank_ids[idx] if iden in valid_ids] for idx in valid_idx]
    unknown = unknown.iloc[valid_idx]
    unknown_ids_conv_count = {k: v for k, v in unknown_ids_conv_count.items() if k in valid_ids}
    num_ids = len(valid_ids)

    res = []

    for top in [1, 5, 10]:
        conv_acc = np.mean([np.any(l[:top]) for l in sim_rank_correct])

        correct_ids = set([target_id for i, l in enumerate(sim_rank_ids) if (target_id := unknown.iloc[i]["identity"]) in l[:top]])
        id_acc = len(correct_ids) / num_ids

        random_guessing = 0.0
        for cnt in unknown_ids_conv_count.values():
            random_guessing += 1 - (1 - min(top, num_ids) / num_ids) ** cnt
        random_guessing /= num_ids

        advantage = id_acc - random_guessing

        # All conversation must be correctly matched
        id_correct_cnt = {}
        for i, l in enumerate(sim_rank_ids):
            target = unknown.iloc[i]["identity"]
            if target in l[:top]:
                id_correct_cnt[target] = id_correct_cnt.get(target, 0) + 1
        correct_ids_all = sum(1 for iden in id_correct_cnt if id_correct_cnt[iden] == unknown_ids_conv_count[iden])
        id_acc_all = correct_ids_all / num_ids

        random_guessing_all = 0.0
        for cnt in unknown_ids_conv_count.values():
            random_guessing_all += (min(top, num_ids) / num_ids) ** cnt
        random_guessing_all /= num_ids

        advantage_all = id_acc_all - random_guessing_all
        
        # print(
        #     f"Top {top} acc: Conv level: {conv_acc:.3f}"
        #     f" | ID level: {id_acc:.3f} (all: {id_acc_all:.3f})"
        #     f" | Random ID: {random_guessing:.3f} (all: {random_guessing_all:.3f})"
        #     f" | Advantage: {advantage:.3f} (all: {advantage_all:.3f})"
        # )

        res.append({
            "sample": sample,
            "top": top,
            "conv_acc": conv_acc,
            "id_acc": id_acc,
            "random_id": random_guessing,
            "advantage": advantage,
            "id_acc_all_conv": id_acc_all,
            "random_id_all_conv": random_guessing_all,
            "advantage_all_conv": advantage_all,
        })

    return res

In [ ]:
random.seed(47)
n_sim = 100
all_res = []

for sample in (list(range(25, num_unique_ids, 25))):
    print(f"========== Sample size: {sample} ==========")
    for _ in tqdm(range(n_sim)):
        res = analyze_wildchat(
            sim_rank_correct,
            sim_rank_ids,
            unknown,
            unknown_ids_conv_count,
            sample=sample,
        )
        all_res.extend(res)

pd.DataFrame(all_res).to_csv(f"wildchat_analysis_{'stylometrix' if use_stylo else 'gemini'}_results.csv", index=False)

In [ ]:
gemini_analysis_df = pd.read_csv("wildchat_analysis_gemini_results.csv")
stylometrix_analysis_df = pd.read_csv("wildchat_analysis_stylometrix_results.csv")

k = 1 # Consider top k matches only
top_k_df = gemini_analysis_df[gemini_analysis_df["top"] == k]
top_k_df["stylometrix"] = stylometrix_analysis_df[stylometrix_analysis_df["top"] == k]["id_acc"]

# Reshape data to long format
df_long = top_k_df.melt(
    id_vars='sample',
    value_vars=['id_acc', "stylometrix", 'random_id'],
    var_name='metric',
    value_name='value',
)

df_long['metric'] = df_long['metric'].map({
    'id_acc': 'gemini-embedding-001',
    "stylometrix": "StyloMetrix",
    'random_id': 'Random guessing',
})

hue_order = df_long['metric'].unique()
n_hue = len(hue_order)

# Map sample → base x positions (0,1,2,...)
x_map = {v: i for i, v in enumerate(order)}

# Width must match your boxplot
width = 0.4

# Compute offsets (same logic seaborn uses)
offsets = np.linspace(-width/2 + 0.1, width/2, n_hue)

# Compute medians (or means if you prefer)
median_df = (
    df_long.groupby(['sample', 'metric'])['value']
    .median()
    .reset_index()
)

# Plot with CI
plt.figure(figsize=(6, 4))
palette = sns.color_palette(n_colors=3)
order = sorted(top_k_df['sample'].unique())
for i, metric in enumerate(hue_order):
    sub = median_df[median_df['metric'] == metric]    
    x = [x_map[s] + offsets[i] for s in sub['sample']]
    y = sub['value'].values
    plt.plot(x, y, label=None, color="silver", linestyle="--")

sns.boxplot(data=df_long, x='sample', y='value', hue='metric', palette=palette, order=order, showfliers=False, width=0.4)

# plt.xticks(sorted(top_k_df['sample'].unique()))
plt.xlabel('')
plt.ylabel('')
plt.tick_params(labelsize=12)
plt.legend(title='', fontsize=12)
plt.grid(linestyle="--", color="lightgray")
plt.tight_layout(pad=0.0)
plt.savefig(f"wildchat_linkage_top_{k}.pdf")
plt.show()

In [ ]:
plt.hist(top_k_df[top_k_df["sample"] == 150]["id_acc"])

In [ ]:
relevant_fields = "conversation"
for i, correct in enumerate(sim_rank_correct):
    if correct[0]:
        print("=== Example ===")
        unknown_data = unknown.iloc[i][relevant_fields]
        known_data = known.iloc[sim_rank[i][0]][relevant_fields]
        print("Unknown:", unknown_data)
        print("Known:", known_data)

In [ ]:
# Same user clustering

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_completeness_v_measure,
)
from sklearn.metrics.pairwise import cosine_similarity


def simulate_adversary(
    unknown_ids: pd.Series,
    unknown_emb: np.ndarray,
    distance_thresholds: np.ndarray = np.arange(0.05, 1.0, 0.05),
) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Simulate a linkage adversary and return results + best predicted labels.
    """
    true_labels = unknown_ids.factorize()[0]
    n_true_clusters = len(unknown_ids.unique())

    distance_matrix = 1.0 - cosine_similarity(unknown_emb)
    np.fill_diagonal(distance_matrix, 0.0)

    results = []
    all_pred_labels = {}

    for threshold in distance_thresholds:
        clustering = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=threshold,
        )
        pred_labels = clustering.fit_predict(distance_matrix)
        all_pred_labels[threshold] = pred_labels

        h, c, v = homogeneity_completeness_v_measure(true_labels, pred_labels)
        results.append(
            {
                "threshold": round(threshold, 3),
                "n_pred_clusters": len(np.unique(pred_labels)),
                "n_true_clusters": n_true_clusters,
                "ari": adjusted_rand_score(true_labels, pred_labels),
                "nmi": normalized_mutual_info_score(true_labels, pred_labels),
                "homogeneity": h,
                "completeness": c,
                "v_measure": v,
            }
        )

    results_df = pd.DataFrame(results)
    best_threshold = results_df.loc[results_df["ari"].idxmax(), "threshold"]
    best_pred_labels = all_pred_labels[best_threshold]

    return results_df, true_labels, best_pred_labels


def per_user_analysis(unknown_ids, true_labels, pred_labels):
    """
    Break down adversary success by user volume.
    """
    df = pd.DataFrame({
        "user_id": unknown_ids.values,
        "true": true_labels,
        "pred": pred_labels,
    })

    user_counts = df.groupby("user_id").size().rename("n_embeddings")

    # Per-user purity: fraction of a user's embeddings in their dominant predicted cluster
    def user_purity(group):
        return group["pred"].value_counts().iloc[0] / len(group)

    # Per-user fragmentation: how many predicted clusters contain this user's embeddings
    def user_fragmentation(group):
        return group["pred"].nunique()

    purity = df.groupby("user_id").apply(user_purity).rename("purity")
    fragmentation = df.groupby("user_id").apply(user_fragmentation).rename("n_clusters_assigned")

    # Per-user: is the dominant cluster "contaminated" by other users?
    def dominant_cluster_purity(group):
        dominant_cluster = group["pred"].value_counts().index[0]
        all_in_cluster = df[df["pred"] == dominant_cluster]
        return (all_in_cluster["user_id"] == group.name).mean()

    cluster_purity = df.groupby("user_id").apply(dominant_cluster_purity).rename("cluster_purity")

    user_stats = pd.concat([user_counts, purity, fragmentation, cluster_purity], axis=1)

    # Bin users by volume
    user_stats["volume_bin"] = pd.cut(
        user_stats["n_embeddings"],
        bins=[0, 2, 5, 10, 25, 50, np.inf],
        labels=["1-2", "3-5", "6-10", "11-25", "26-50", "50+"],
    )

    summary = user_stats.groupby("volume_bin", observed=True).agg(
        n_users=("purity", "size"),
        total_embeddings=("n_embeddings", "sum"),
        mean_purity=("purity", "mean"),
        median_purity=("purity", "median"),
        mean_fragmentation=("n_clusters_assigned", "mean"),
        mean_cluster_purity=("cluster_purity", "mean"),
    ).round(3)

    all_row = pd.DataFrame(
        {
            "n_users": [len(user_stats)],
            "total_embeddings": [user_stats["n_embeddings"].sum()],
            "mean_purity": [user_stats["purity"].mean()],
            "median_purity": [user_stats["purity"].median()],
            "mean_fragmentation": [user_stats["n_clusters_assigned"].mean()],
            "mean_cluster_purity": [user_stats["cluster_purity"].mean()],
        },
        index=pd.CategoricalIndex(["all"], name="volume_bin"),
    ).round(3)

    summary = pd.concat([summary, all_row])

    return user_stats, summary


def pair_weight_analysis(unknown_ids, true_labels, pred_labels):
    """
    Show how much each volume bin contributes to the global ARI
    by computing the share of concordant/discordant pairs.
    """
    df = pd.DataFrame({
        "user_id": unknown_ids.values,
        "true": true_labels,
        "pred": pred_labels,
    })

    user_counts = df.groupby("user_id").size()

    # Number of same-user pairs per user: C(n, 2)
    pairs_per_user = (user_counts * (user_counts - 1)) / 2
    total_same_pairs = pairs_per_user.sum()

    bins = pd.cut(
        user_counts,
        bins=[0, 2, 5, 10, 25, 50, np.inf],
        labels=["1-2", "3-5", "6-10", "11-25", "26-50", "50+"],
    )

    pair_share = pairs_per_user.groupby(bins, observed=True).sum() / total_same_pairs
    pair_share = pair_share.rename("share_of_same_user_pairs").round(3)

    user_share = bins.value_counts(normalize=True).rename("share_of_users").round(3)

    comparison = pd.concat([user_share, pair_share], axis=1).sort_index()
    return comparison


# --- Run everything ---
results_df, true_labels, best_pred_labels = simulate_adversary(unknown_ids, unknown_emb)

best = results_df.loc[results_df["ari"].idxmax()]
print("=" * 60)
print("BEST ADVERSARY RESULT (by ARI)")
print("=" * 60)
print(best.to_string())
print()

user_stats, summary = per_user_analysis(unknown_ids, true_labels, best_pred_labels)
print("=" * 60)
print("PER-USER ANALYSIS BY VOLUME BIN")
print("=" * 60)
print(summary.to_string())
print()
print("Columns:")
print("  mean_purity          : avg fraction of user's embeddings in their dominant cluster (1.0 = no splitting)")
print("  mean_fragmentation   : avg number of clusters a user's embeddings land in (1.0 = all together)")
print("  mean_cluster_purity  : avg purity of the dominant cluster (1.0 = no contamination from other users)")
print()

pair_weights = pair_weight_analysis(unknown_ids, true_labels, best_pred_labels)
print("=" * 60)
print("PAIR WEIGHT ANALYSIS (how much each bin drives ARI)")
print("=" * 60)
print(pair_weights.to_string())
print()
print("If share_of_same_user_pairs >> share_of_users, that bin")
print("disproportionately drives the global ARI score.")
print()

# Detailed look at the tails
print("=" * 60)
print("MOST EXPOSED USERS (top 10 by embedding count)")
print("=" * 60)
top_users = user_stats.nlargest(10, "n_embeddings")
print(top_users[["n_embeddings", "purity", "n_clusters_assigned", "cluster_purity"]].to_string())
print()

print("=" * 60)
print("LIGHT USERS (1-2 embeddings) SAMPLE")
print("=" * 60)
light = user_stats[user_stats["n_embeddings"] <= 2]
print(f"Count: {len(light)} users")
print(f"Mean cluster purity: {light['cluster_purity'].mean():.3f}")
print(f"(Interpretation: when a light user lands in a cluster, what fraction of")
print(f" that cluster belongs to them? Low = they're buried among other users)")